# tsfresh feature generation and t-sne analysis on StressID and ExpData datasets

In [6]:
import os
os.environ['OMP_NUM_THREADS'] = "1"
os.environ['MKL_NUM_THREADS'] = "1"
os.environ['OPENBLAS_NUM_THREADS'] = "1"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

from tsfresh import extract_features, select_features
from tsfresh.feature_selection.relevance import calculate_relevance_table
from tsfresh.feature_extraction.settings import EfficientFCParameters, MinimalFCParameters, IndexBasedFCParameters
from tsfresh.utilities.dataframe_functions import impute

from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

BASE_PATH = "../../.."
DATASET_SEPARATOR = ";"
DATASET = f"{BASE_PATH}/experiment-data"
DATASET_FILENAME = f"{DATASET}/flat_dataset.alltasks.csv.gz"
DATASET_SUBJECTNORM_FILENAME = f"{DATASET}/flat_dataset.alltasks.subjnorm.csv.gz"
LABELS_SEPARATOR = ","
LABELS = f"{DATASET}/labels.csv"

EXTRACT = True
SELECT = True
EXTRACTED_FEATURES_FILENAME = f"{DATASET}/tsfresh_extfeat.csv.gz"
EXTRACTED_ALLFEATURES_FILENAME = f"{DATASET}/tsfresh_all_extfeat.csv.gz"
SELECTED_FEATURES_FILENAME = f"{DATASET}/tsfresh_selfeat.csv.gz"

BIN_LABELS = ["NoStress", "Stress"]
TER_LABELS = ["Relaxed", "Stress", "RealStress"]
QAD_LABELS = ["Relaxed", "Stress", "RealStress", "Amused"]

LABELS_CONF = {
    "b": {
        "col_name": "binary-stress",
        "enabled": True,
        "stratification": True,
        "classes": BIN_LABELS,
    },
    "t": {
        "col_name": "affect3-class",
        "enabled": True,
        "stratification": True,
        "classes": TER_LABELS,
    },
    "q": {
        "col_name": "affect4-class",
        "enabled": True,
        "stratification": True,
        "classes": QAD_LABELS,
    },
}

In [7]:
class FullData:
    def __init__(self):
        self.subject: list[int] = []
        self.task: list[str] = []
        self.timestamp: list[float] = []
        self.eda: list[float] = []
        self.ppg: list[float] = []
        self.accel_x: list[float] = []
        self.accel_y: list[float] = []
        self.accel_z: list[float] = []
        self.gyro_x: list[float] = []
        self.gyro_y: list[float] = []
        self.gyro_z: list[float] = []
        self.temp: list[float] = []
        self.pressure: list[float] = []
        self.b_classes: list[int] = []
        self.t_classes: list[int] = []
        self.q_classes: list[int] = []

    def to_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(
            {
                "Subject": self.subject,
                "Task": self.task,
                "Timestamp": self.timestamp,
                "EDA": self.eda,
                "PPG": self.ppg,
                "Accel_X": self.accel_x,
                "Accel_Y": self.accel_y,
                "Accel_Z": self.accel_z,
                "Gyro_X": self.gyro_x,
                "Gyro_Y": self.gyro_y,
                "Gyro_Z": self.gyro_z,
                "Temperature": self.temp,
                "Pressure": self.pressure,
                "Bin_Class": self.b_classes,
                "Ter_Class": self.t_classes,
                "Qad_Class": self.q_classes,
            }
        )

    def assert_lengths(self) -> None:
        differs = False
        lengths = {}
        expected_len = len(self.timestamp)
        # print(f"Asserting the length of {expected_len} items on the columns")
        for name, instance_attr in self.__dict__.items():
            lengths[name] = len(instance_attr)
            if len(instance_attr) != expected_len:
                differs = True
        if differs:
            AssertionError(f"Columns are not the same lenght: {lengths}")


### Build dataset from data files

In [8]:
# Creating labels object

labels_df = pd.read_csv(LABELS, sep=LABELS_SEPARATOR, header=0, index_col=0)
tasks: dict[str, pd.Series] = {}
for key, conf in LABELS_CONF.items():
    if conf["enabled"]:
        tasks[key] = labels_df[conf["col_name"]]

display(tasks["b"])
tasks_with_labels: list[str] = tasks["b"].index


subject/task
01-AmusementClip    0
01-Baseline         0
01-EmoReset         0
01-FormL            1
01-FormM            1
                   ..
21-Baseline         0
21-EmoReset         0
21-FormL            0
21-FormM            0
21-StressClip       1
Name: binary-stress, Length: 126, dtype: int64

In [9]:
# Creating dataset file, it may be skipped

CREATE_DATASET_FILE = True
USE_UNCALIBRATED_SIGNALS = False
STANDARIZATION_BYTASK = False
STANDARIZATION_BYSUBJECT = True
NEUROCLEAN = True
EXPECTED_NUM_FILES = 21
SAMPLING_RATE = 51.2
CREATE_TEST = False
TEST_N_ITERATIONS = 3

if CREATE_DATASET_FILE:
    import glob
    import neurokit2 as nk
    type FeatureDict = dict[str, np.ndarray]

    eda_col = "Skin_Conductance_Uncal" if USE_UNCALIBRATED_SIGNALS else "Skin_Conductance"
    ppg_col = "PPG_Uncal" if USE_UNCALIBRATED_SIGNALS else "PPG"

    col_types = {
        "Timestamp": float,
        "Event": str,
        "ExtraEvent": str,
        "AccelLN_X": float,
        "AccelLN_Y": float,
        "AccelLN_Z": float,
        "Battery": float,
        "GSR_Range": int,
        "Skin_Conductance": float,
        "Skin_Resistance": float,
        "Gyro_X": float,
        "Gyro_Y": float,
        "Gyro_Z": float,
        "PPG": float,
        "Pressure": float,
        "Temperature": float,
        "AccelLN_X_Uncal": int,
        "AccelLN_Y_Uncal": int,
        "AccelLN_Z_Uncal": int,
        "Skin_Conductance_Uncal": int,
        "PPG_Uncal": int,
    }

    ####### LOAD DATA
    filelist = glob.glob(f"{DATASET}/*.Annotated.csv")
    filelist.sort()
    if len(filelist) != EXPECTED_NUM_FILES:
        raise ValueError(f"Expected {EXPECTED_NUM_FILES} files, found: {len(filelist)}")

    full_data = FullData()
    subject_to_int: dict[str, int] = {}
    subject_counter = 0

    def split_by_task(df: pd.DataFrame) -> list[tuple[str, pd.DataFrame]]:
        output: list[tuple[str, pd.DataFrame]] = []
        tasks = ["Baseline", "AmusementClip", "StressClip", "EmoReset", "FormL", "FormM", "Debriefing"]
        start_idx = end_idx = 0
        for task in tasks:
            if task == "FormL":
                if "FormLRead" in df["Event"].values:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormLRead"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "L15"].index[-1])
                else:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormL"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "FormL"].index[-1])
            elif task == "FormM":
                if "FormMRead" in df["Event"].values:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormMRead"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "M15"].index[-1])
                else:
                    start_idx = df.index.get_loc(df[df["Event"] == "FormM"].index[0])
                    end_idx = df.index.get_loc(df[df["Event"] == "FormM"].index[-1])
            else:
                start_idx = df.index.get_loc(df[df["Event"] == task].index[0])
                end_idx = df.index.get_loc(df[df["Event"] == task].index[-1])
            output.append((task, df[start_idx:end_idx]))
        return output

    iteration_counter = 0
    for item in filelist:
        file: pd.DataFrame = pd.read_csv(
            item,
            delimiter=";",
            date_format=r"%Y-%m-%d %H:%M:%S.%f",
            parse_dates=["Datetime", "Timestamp"],
            index_col=["Datetime"],
            dtype=col_types,
        )
        filename = item.split("/")[-1]
        subject_id = filename.split("-")[1]
        if subject_id not in subject_to_int:
            subject_to_int[subject_id] = subject_counter
            subject_counter += 1

        for task, event_df in split_by_task(file):
            label_id = f"{subject_id}-{task}"
            if label_id not in tasks_with_labels:
                continue
            n_elements = event_df["Timestamp"].size
            ppg_out = []
            ppg_signal = np.array(
                nk.ppg_clean(np.array(event_df[ppg_col]), sampling_rate=SAMPLING_RATE)
                if NEUROCLEAN
                else event_df[ppg_col].to_list()
            )
            eda_out = []
            eda_signal = np.array(
                nk.eda_clean(
                    np.array(event_df[eda_col]), sampling_rate=SAMPLING_RATE, method="neurokit"
                )
                if NEUROCLEAN
                else event_df[eda_col].to_list()
            )
            if ppg_signal.size != n_elements:
                raise Exception(f"Sizes differ {ppg_signal.size} vs {n_elements}")
            if STANDARIZATION_BYTASK:
                ppg_out = (ppg_signal - ppg_signal.mean()) / ppg_signal.std()
                eda_out = (eda_signal - eda_signal.mean()) / eda_signal.std()
            else:
                ppg_out = ppg_signal.copy()
                eda_out = eda_signal.copy()
            full_data.subject.extend(np.full(n_elements, subject_to_int[subject_id]))
            full_data.task.extend(np.full(n_elements, task))
            full_data.timestamp.extend(event_df["Timestamp"].to_list())
            full_data.ppg.extend(ppg_out)
            full_data.eda.extend(eda_out)
            full_data.accel_x.extend(event_df["AccelLN_X"].to_list())
            full_data.accel_y.extend(event_df["AccelLN_Y"].to_list())
            full_data.accel_z.extend(event_df["AccelLN_Z"].to_list())
            full_data.gyro_x.extend(event_df["Gyro_X"].to_list())
            full_data.gyro_y.extend(event_df["Gyro_Y"].to_list())
            full_data.gyro_z.extend(event_df["Gyro_Z"].to_list())
            full_data.temp.extend(event_df["Temperature"].to_list())
            full_data.pressure.extend(event_df["Pressure"].to_list())
            full_data.b_classes.extend(np.full(n_elements, tasks["b"][label_id]))
            full_data.t_classes.extend(np.full(n_elements, tasks["t"][label_id]))
            full_data.q_classes.extend(np.full(n_elements, tasks["q"][label_id]))
            full_data.assert_lengths()

        iteration_counter += 1
        if CREATE_TEST and iteration_counter >= TEST_N_ITERATIONS:
            break

    fulldata_df = full_data.to_dataframe()
    if STANDARIZATION_BYSUBJECT:
        for subject_int in range(subject_counter):
            mask = fulldata_df["Subject"] == subject_int
            fulldata_df.loc[mask, "EDA"] = (fulldata_df[mask]["EDA"] - fulldata_df[mask]["EDA"].mean()) / fulldata_df[mask]["EDA"].std()
            fulldata_df.loc[mask, "PPG"] = (fulldata_df[mask]["PPG"] - fulldata_df[mask]["PPG"].mean()) / fulldata_df[mask]["PPG"].std()

    if not CREATE_TEST:
        fulldata_df.to_csv(DATASET_SUBJECTNORM_FILENAME, sep=DATASET_SEPARATOR, index=False, compression="gzip")
    display(fulldata_df)


,Subject,Task,Timestamp,EDA,PPG,Accel_X,Accel_Y,Accel_Z,Gyro_X,Gyro_Y,Gyro_Z,Temperature,Pressure,Bin_Class,Ter_Class,Qad_Class
0,0,Baseline,1749609303080.02,3.339991,-0.064332,8.076087,-3.836957,3.500000,-63.847328,7.557252,59.007634,28.093986,99.874754,0,0,0
1,0,Baseline,1749609303099.55,3.361933,-0.130831,7.978261,-3.902174,3.576087,-61.541985,12.748092,59.725191,28.093986,99.872008,0,0,0
2,0,Baseline,1749609303119.08,3.382377,-0.195255,7.967391,-4.119565,3.565217,-61.129771,17.404580,60.320611,28.093986,99.869262,0,0,0
3,0,Baseline,1749609303138.61,3.399863,-0.254358,7.836957,-4.163043,3.619565,-57.541985,18.351145,58.748092,28.093986,99.877500,0,0,0
4,0,Baseline,1749609303158.14,3.413235,-0.305000,7.554348,-4.217391,3.619565,-54.305344,19.312977,56.167939,28.093986,99.877500,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1435016,20,FormM,1752561365895.54,0.793654,0.107725,8.576087,6.217391,7.304348,-15.389313,3.679389,2.961832,29.905178,98.695416,0,0,0
1435017,20,FormM,1752561365915.07,0.797666,0.012858,8.576087,6.250000,7.326087,-15.404580,3.496183,2.580153,29.905178,98.698168,0,0,0
1435018,20,FormM,1752561365934.6,0.801924,-0.064151,8.619565,6.271739,7.315217,-15.083969,3.648855,2.427481,29.905178,98.703673,0,0,0
1435019,20,FormM,1752561365954.13,0.806349,-0.113136,8.619565,6.260870,7.271739,-15.633588,3.984733,2.427481,29.905178,98.700921,0,0,0


In [6]:
DESIRED_COLUMNS = ["Timestamp", "EDA", "PPG"]
DESIRED_LABELS = "Bin_Class"

In [ ]:
if EXTRACT:
    fulldata_df = pd.read_csv(DATASET_SUBJECTNORM_FILENAME, sep=DATASET_SEPARATOR, compression="gzip")
    X = fulldata_df.loc[:, DESIRED_COLUMNS]
    y = fulldata_df.loc[:, DESIRED_LABELS]
    X["Subject-Task"] = fulldata_df.apply(lambda row: f"{row['Subject']+1:02d}-{row['Task']}", axis=1)
    X_groups = fulldata_df["Subject"]

In [ ]:
if EXTRACT:
    display(X)
    display(y)
    display(X_groups)
    samples_stats_df = X.groupby(["Subject-Task"]).agg(samples=("Timestamp", "size"))
    samples_stats_df["Duration(min)"] = samples_stats_df["samples"] / SAMPLING_RATE / 60.0
    display(samples_stats_df)

In [ ]:
if EXTRACT:
    features = extract_features(
        X,
        # y=tasks["b"],
        column_id="Subject-Task",
        column_sort="Timestamp",
        impute_function=impute,
        default_fc_parameters=EfficientFCParameters(),
        n_jobs=6,
    )

In [9]:
if EXTRACT:
    features.to_csv(EXTRACTED_FEATURES_FILENAME, sep=DATASET_SEPARATOR, compression="gzip")
    display(features)
if SELECT:
    features = pd.read_csv(EXTRACTED_FEATURES_FILENAME, index_col=0, sep=DATASET_SEPARATOR, compression="gzip")
    display(features)

# Selecting rows that actually have entries in "labels" file
idx = list(features.merge(tasks["b"], left_index=True, right_index=True).index)
y = tasks["b"].loc[idx]
x = features.loc[idx]
display(y)
display(x)

,PPG__variance_larger_than_standard_deviation,PPG__has_duplicate_max,PPG__has_duplicate_min,PPG__has_duplicate,PPG__sum_values,PPG__abs_energy,PPG__mean_abs_change,PPG__mean_change,PPG__mean_second_derivative_central,PPG__median,...,EDA__fourier_entropy__bins_5,EDA__fourier_entropy__bins_10,EDA__fourier_entropy__bins_100,EDA__permutation_entropy__dimension_3__tau_1,EDA__permutation_entropy__dimension_4__tau_1,EDA__permutation_entropy__dimension_5__tau_1,EDA__permutation_entropy__dimension_6__tau_1,EDA__permutation_entropy__dimension_7__tau_1,EDA__query_similarity_count__query_None__threshold_0.0,EDA__mean_n_absolute_max__number_of_maxima_7
01-AmusementClip,0.0,0.0,0.0,0.0,1.492140e-13,9216.0,0.173736,-2.066180e-05,2.135337e-05,-0.239122,...,0.045395,0.090729,0.181214,0.521230,0.658779,0.805258,0.959074,1.118652,0.0,2.140503
01-Baseline,1.0,0.0,0.0,0.0,4.973799e-14,12288.0,0.165945,1.344990e-05,-9.243722e-06,-0.247115,...,0.090729,0.136002,0.459635,0.534750,0.668079,0.807880,0.954546,1.109999,0.0,3.520089
01-EmoReset,0.0,0.0,0.0,0.0,-4.884981e-14,9216.0,0.176090,4.725521e-07,-4.452169e-07,-0.248224,...,0.045395,0.125256,0.181214,0.596761,0.770221,0.952638,1.145402,1.348162,0.0,2.331208
01-FormL,1.0,0.0,0.0,0.0,1.652012e-13,15360.0,0.177607,-1.704256e-05,3.850086e-06,-0.249488,...,0.136002,0.215617,0.452944,0.578573,0.641838,0.709142,0.779708,0.852263,0.0,2.978883
01-FormM,0.0,0.0,0.0,0.0,-2.131628e-13,18432.0,0.176206,-9.952111e-06,-6.019823e-06,-0.216350,...,0.136002,0.215617,0.531534,0.546509,0.663437,0.786794,0.917983,1.054484,0.0,3.322106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21-Baseline,1.0,0.0,0.0,0.0,1.678657e-13,9574.0,0.216214,1.345987e-05,3.161732e-06,-0.413190,...,0.090729,0.136002,0.136002,0.551855,0.619738,0.690555,0.765160,0.841431,0.0,2.779699
21-EmoReset,0.0,0.0,0.0,0.0,-9.903189e-14,9522.0,0.210997,-1.389837e-05,7.515393e-06,-0.367913,...,0.090729,0.136002,0.181214,0.625014,0.709318,0.796201,0.888106,0.982514,0.0,2.948996
21-FormL,0.0,0.0,0.0,0.0,1.421085e-14,12389.0,0.197080,5.312024e-06,-9.589858e-06,-0.354683,...,0.125256,0.170467,0.316475,0.563991,0.650301,0.741734,0.836473,0.934408,0.0,2.592204
21-FormM,1.0,0.0,0.0,0.0,1.580958e-13,20274.0,0.209753,-8.422849e-06,-1.318382e-07,-0.378217,...,0.125256,0.125256,0.181214,0.671276,0.748316,0.829156,0.912573,0.999499,0.0,4.227752


subject/task
01-AmusementClip    0
01-Baseline         0
01-EmoReset         0
01-FormL            1
01-FormM            1
                   ..
21-Baseline         0
21-EmoReset         0
21-FormL            0
21-FormM            0
21-StressClip       1
Name: binary-stress, Length: 126, dtype: int64

,PPG__variance_larger_than_standard_deviation,PPG__has_duplicate_max,PPG__has_duplicate_min,PPG__has_duplicate,PPG__sum_values,PPG__abs_energy,PPG__mean_abs_change,PPG__mean_change,PPG__mean_second_derivative_central,PPG__median,...,EDA__fourier_entropy__bins_5,EDA__fourier_entropy__bins_10,EDA__fourier_entropy__bins_100,EDA__permutation_entropy__dimension_3__tau_1,EDA__permutation_entropy__dimension_4__tau_1,EDA__permutation_entropy__dimension_5__tau_1,EDA__permutation_entropy__dimension_6__tau_1,EDA__permutation_entropy__dimension_7__tau_1,EDA__query_similarity_count__query_None__threshold_0.0,EDA__mean_n_absolute_max__number_of_maxima_7
01-AmusementClip,0.0,0.0,0.0,0.0,1.492140e-13,9216.0,0.173736,-2.066180e-05,2.135337e-05,-0.239122,...,0.045395,0.090729,0.181214,0.521230,0.658779,0.805258,0.959074,1.118652,0.0,2.140503
01-Baseline,1.0,0.0,0.0,0.0,4.973799e-14,12288.0,0.165945,1.344990e-05,-9.243722e-06,-0.247115,...,0.090729,0.136002,0.459635,0.534750,0.668079,0.807880,0.954546,1.109999,0.0,3.520089
01-EmoReset,0.0,0.0,0.0,0.0,-4.884981e-14,9216.0,0.176090,4.725521e-07,-4.452169e-07,-0.248224,...,0.045395,0.125256,0.181214,0.596761,0.770221,0.952638,1.145402,1.348162,0.0,2.331208
01-FormL,1.0,0.0,0.0,0.0,1.652012e-13,15360.0,0.177607,-1.704256e-05,3.850086e-06,-0.249488,...,0.136002,0.215617,0.452944,0.578573,0.641838,0.709142,0.779708,0.852263,0.0,2.978883
01-FormM,0.0,0.0,0.0,0.0,-2.131628e-13,18432.0,0.176206,-9.952111e-06,-6.019823e-06,-0.216350,...,0.136002,0.215617,0.531534,0.546509,0.663437,0.786794,0.917983,1.054484,0.0,3.322106
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21-Baseline,1.0,0.0,0.0,0.0,1.678657e-13,9574.0,0.216214,1.345987e-05,3.161732e-06,-0.413190,...,0.090729,0.136002,0.136002,0.551855,0.619738,0.690555,0.765160,0.841431,0.0,2.779699
21-EmoReset,0.0,0.0,0.0,0.0,-9.903189e-14,9522.0,0.210997,-1.389837e-05,7.515393e-06,-0.367913,...,0.090729,0.136002,0.181214,0.625014,0.709318,0.796201,0.888106,0.982514,0.0,2.948996
21-FormL,0.0,0.0,0.0,0.0,1.421085e-14,12389.0,0.197080,5.312024e-06,-9.589858e-06,-0.354683,...,0.125256,0.170467,0.316475,0.563991,0.650301,0.741734,0.836473,0.934408,0.0,2.592204
21-FormM,1.0,0.0,0.0,0.0,1.580958e-13,20274.0,0.209753,-8.422849e-06,-1.318382e-07,-0.378217,...,0.125256,0.125256,0.181214,0.671276,0.748316,0.829156,0.912573,0.999499,0.0,4.227752


In [12]:
if SELECT:
    print(f"Selected {len(x)} entries from X and {len(y)} labels.")
    # Asserting order of labels
    for x_idx, y_idx, real in zip(x.index, y.index, tasks["b"].index):
        assert x_idx == y_idx and y_idx == real, "There was a change of order between tasks[b] labels and x's index"

    # relevance = calculate_relevance_table(x, y)
    # display(relevance)
    # relevance.to_clipboard()
    X_selected = select_features(x, y, fdr_level=0.1)
    display(X_selected)

Selected 126 entries from X and 126 labels.


""
01-AmusementClip
01-Baseline
01-EmoReset
01-FormL
01-FormM
...
21-Baseline
21-EmoReset
21-FormL
21-FormM


In [ ]:
# NO RELEVANT FEATURES FOUND
if SELECT:
    X_selected.to_csv(SELECTED_FEATURES_FILENAME)
else:
    X_selected = pd.read_csv(SELECTED_FEATURES_FILENAME, index_col=0)
display(X_selected)

In [ ]:
# RFECV for ExpData
RANDOM_STATE = 21
SPLITS = 10

estimator = RandomForestClassifier(max_depth=5, random_state=RANDOM_STATE)
cv = StratifiedKFold(n_splits=SPLITS, shuffle=True, random_state=RANDOM_STATE)
selector = RFECV(estimator=estimator, step=2, cv=cv, scoring="balanced_accuracy")
selector.fit(x, y)
features_mask = selector.support_
x_rfe = x.loc[:, features_mask]

features_scores = {"scores": [], "features": []}
for score, feat in zip(selector.estimator_.feature_importances_, x_rfe.columns):
    features_scores["scores"].append(score)
    features_scores["features"].append(feat)
features_scores_df = pd.DataFrame(features_scores).sort_values(by="scores", ascending=False)
features_scores_df = features_scores_df.reset_index(drop=True)
features_scores_df.index = features_scores_df.index + 1
print("Selected features scores for ExpData")
display(features_scores_df)


Selected features scores for ExpData


,scores,features
1,0.015628,"PPG__agg_linear_trend__attr_""stderr""__chunk_le..."
2,0.013228,PPG__count_above_mean
3,0.012568,EDA__number_cwt_peaks__n_5
4,0.011165,PPG__ratio_beyond_r_sigma__r_3
5,0.009775,"EDA__change_quantiles__f_agg_""mean""__isabs_Tru..."
...,...,...
1508,0.000000,"PPG__agg_linear_trend__attr_""slope""__chunk_len..."
1509,0.000000,PPG__cwt_coefficients__coeff_11__w_20__widths_...
1510,0.000000,PPG__cwt_coefficients__coeff_12__w_2__widths_(...
1511,0.000000,PPG__cwt_coefficients__coeff_12__w_5__widths_(...


In [ ]:
# Divergence analysis on StressID
import plotly.express as px

perplexity = np.arange(15, 210, 15)
divergence = []
si_Ncomp = 3

for i in perplexity:
    model = TSNE(n_components=si_Ncomp, init="pca", perplexity=i)
    reduced = model.fit_transform(X_selected)
    divergence.append(model.kl_divergence_)

fig = px.line(x=perplexity, y=divergence, markers=True, width=800, height=600)
fig.update_layout(xaxis_title="Perplexity Values", yaxis_title="KL Divergence")
fig.update_traces(line_color="red", line_width=1)
fig.show()

In [ ]:
# t-SNE in StressID
# Best N-comp=2, Perp=190 np.arange(10, 200, 10)
si_Ncomp = 2
si_Perp = 190

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(X_selected)
display(si_tsne.kl_divergence_)

fig = px.scatter(x=si_X_tsne[:,0], y=si_X_tsne[:,1], color=y, width=800, height=600)
fig.update_layout(
    title="t-SNE visualization of StressID dataset",
    xaxis_title="1st t-SNE",
    yaxis_title="2nd t-SNE",
)
fig.show()

In [ ]:
# t-SNE in StressID
# Best N-comp=3, Perp=90 np.arange(15, 210, 15)
si_Ncomp = 3
si_Perp = 90

si_tsne = TSNE(n_components=si_Ncomp, perplexity=si_Perp, random_state=RAN_STATE)
si_X_tsne = si_tsne.fit_transform(X_selected)
display(si_tsne.kl_divergence_)

fig = px.scatter_3d(x=si_X_tsne[:, 0], y=si_X_tsne[:, 1], z=si_X_tsne[:,2], color=y, opacity=0.7, width=800, height=600)
fig.update_layout(title="t-SNE visualization of StressID dataset")
fig.show()